This notebook contains the final training pipeline for the waste classification model.
I previously tested Logistic Regression and Neural Networks. Although these models showed high accuracy during training, they performed poorly on real-world camera data because they lacked spatial awareness.

In [ ]:
import os, json
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, Conv2D, MaxPooling2D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

BASE_DIR = "/content/data"

DATA_DIR  = os.path.join(BASE_DIR, "augmented")
CSV_PATH  = os.path.join(BASE_DIR, "dataset.csv")
CNN_PATH  = os.path.join(BASE_DIR, "model_cnn.keras")
META_PATH = os.path.join(BASE_DIR, "model_meta.json")

CATEGORIES = ["plastic", "paper", "glass", "bio", "mixed"]
CATEGORY_NAMES = {
    "plastic": "Plastic (yellow bin)",
    "paper": "Paper (blue bin)",
    "glass": "Glass (green bin)",
    "bio": "Bio (brown bin)",
    "mixed": "Mixed (black bin)",
}
N_CLASSES = len(CATEGORIES)
IMG_SIZE = 128

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")
FEATURE_NAMES = [c for c in df.columns if c not in ("category", "filename")]

print("Loading images")
per_cat = {}
for idx, cat in enumerate(CATEGORIES):
    cat_dir = os.path.join(DATA_DIR, cat)
    if not os.path.exists(cat_dir):
        print(f"  Warning: {cat} folder not found!")
        per_cat[cat] = []
        continue
    imgs = []
    for fname in os.listdir(cat_dir):
        if not fname.lower().endswith((".jpg", ".jpeg")):
            continue
        try:
            img = Image.open(os.path.join(cat_dir, fname)).convert("RGB")
            img = img.resize((IMG_SIZE, IMG_SIZE))
            arr = np.asarray(img).astype("float32") / 255
            if arr.shape == (IMG_SIZE, IMG_SIZE, 3):
                imgs.append((arr, idx))
        except:
            continue
    per_cat[cat] = imgs
    print(f"  {cat}: {len(imgs)} images")

counts = {cat: len(per_cat[cat]) for cat in CATEGORIES}
valid_counts = [c for c in counts.values() if c > 0]
if not valid_counts:
    raise ValueError("No images found! Check if 'data.zip' was unzipped correctly.")

min_count = min(valid_counts)
print(f"\nBalancing each category to {min_count}")

all_data = []
rng = np.random.RandomState(42)
for cat in CATEGORIES:
    items = per_cat[cat]
    if len(items) == 0:
        continue
    indices = rng.choice(len(items), size=min(min_count, len(items)), replace=False)
    for i in indices:
        all_data.append(items[i])

rng.shuffle(all_data)
X = np.array([d[0] for d in all_data])
y = np.array([d[1] for d in all_data])
print(f"Total: {len(X)} images ({min_count} per category)")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
y_tr_cat = to_categorical(y_tr, N_CLASSES)
y_te_cat = to_categorical(y_te, N_CLASSES)

cw = compute_class_weight("balanced", classes=np.unique(y_tr), y=y_tr)
class_weight = {i: w for i, w in enumerate(cw)}

cnn = Sequential([
    Conv2D(32, kernel_size=(3, 3), input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(64, kernel_size=(3, 3)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Conv2D(128, kernel_size=(3, 3)),
    BatchNormalization(),
    Activation("relu"),
    MaxPooling2D(pool_size=(2, 2)),

    Flatten(),
    Dense(256),
    BatchNormalization(),
    Activation("relu"),
    Dropout(0.5),
    Dense(N_CLASSES, activation="softmax")
])

cnn.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
print(cnn.summary())

cnn.fit(
    X_tr, y_tr_cat,
    batch_size=32,
    epochs=15,
    verbose=1,
    validation_data=(X_te, y_te_cat),
    class_weight=class_weight,
    callbacks=[EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)],
)

y_pred = cnn.predict(X_te).argmax(axis=1)
y_true = y_te_cat.argmax(axis=1)
acc = accuracy_score(y_true, y_pred)

print(f"\n{'='*60}\n  CNN Accuracy: {acc*100:.1f}%\n{'='*60}")
print(classification_report(y_true, y_pred, target_names=CATEGORIES, digits=3))

cnn.save(CNN_PATH)
meta = {
    "categories": CATEGORIES,
    "category_names": CATEGORY_NAMES,
    "feature_names": FEATURE_NAMES,
    "img_size": IMG_SIZE,
    "accuracy_cnn": round(acc, 4),
    "n_total": len(X),
}
with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f"\nModel saved to: {CNN_PATH}")